# Central univariate Cox analysis

Single notebook covering all univariate result sets:

| Label | Description |
|---|---|
| DFCI | RHINO federated site — Dana-Farber |
| JHU | RHINO federated site — Johns Hopkins |
| MSK | RHINO federated site — Memorial Sloan Kettering |
| PROFILE | PROFILE cohort, first-treatment anchor |
| ARPI-adjusted | PROFILE cohort, first-ARPI anchor |
| Gleason | PROFILE cohort, first-treatment anchor + Gleason covariate |
| Gleason+ARPI | PROFILE cohort, first-ARPI anchor + Gleason covariate |

**Sections**
1. Imports, paths, category mapping
2. Data loaders + unified schema
3. Volcano figures (one per source)
4. Androgen-axis forest plots (per-source + cross-center comparison)

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

# helpers.plotting lives in the server-side code repo
sys.path.insert(0, "/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/survival_analysis")
from helpers.plotting import canonicalize_lab_name

## 2. Paths

In [ ]:
SURVIVAL_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
RHINO_DIR     = SURVIVAL_ROOT / "CAIA/RHINO_runs"
PROFILE_ROOT  = SURVIVAL_ROOT / "PROFILE"
OUT_DIR       = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS/univariate_central")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CODE_DIR     = Path("/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/survival_analysis")
OMOP_MAP_CSV = CODE_DIR / "OMOP_to_DFCI_lab_ids.csv"

LANDMARKS = [0, 90]
ENDPOINT  = "platinum"

## 3. Clinical category mapping

In [ ]:
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
ANDROGEN = {"PSA", "Testosterone"}
OTHER = {"TSH"}
DROP = {"Body height"}

CATEGORY_MAP = {}
for _label, _members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Vitals", VITALS), ("Androgen axis", ANDROGEN), ("Other", OTHER),
]:
    for m in _members:
        CATEGORY_MAP[m] = _label

DRAW_ORDER   = ["Other", "Vitals", "CMP", "LFT", "CBC", "Androgen axis"]
LEGEND_ORDER = ["Androgen axis", "CBC", "LFT", "CMP", "Vitals", "Other"]

CATEGORY_COLORS = {
    "Androgen axis": "#8e1c2b",
    "CBC":           "#16a085",
    "LFT":           "#e67e22",
    "CMP":           "#7d3c98",
    "Vitals":        "#5d6d7e",
    "Other":         "#95a5a6",
}
NS_COLOR = "#d5d8dc"


def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(canonicalize_lab_name(lab_name), "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


def parse_feature(name):
    if "__" in name:
        lab, stat = name.split("__", 1)
        return lab, stat
    return name, ""

## 4. Data loaders

Both loaders return a DataFrame with a unified schema:

`source, landmark_days, endpoint, lab_name, lab_name_raw, feature_stat,
n_patients, n_events, coef_feature, hazard_ratio_per_sd, ci_lower, ci_upper,
p_value, q_value, note`

Key normalizations:
- RHINO: `landmark_day → landmark_days`; `coef_feature = log(hr_per_sd)`
- RHINO: `lab_name` canonicalized via OMOP map (`OMOP_to_DFCI_lab_ids.csv`) with `canonicalize_lab_name` as fallback
- PROFILE: `hazard_ratio_per_sd = exp(coef_feature)` if missing

In [ ]:
import re as _re


def _build_omop_name_map(csv_path: Path) -> dict:
    """Return {rhino_lab_name: canonical_name} from OMOP_to_DFCI_lab_ids.csv.

    RHINO lab_name encodes the OMOP concept name with:
      " [" → "__", "] " → "__", "/" → "_", " " → "_"
    e.g. "Prostate specific Ag [Mass/volume] in Serum or Plasma"
      → "Prostate_specific_Ag__Mass_volume__in_Serum_or_Plasma"
    """
    try:
        df = pd.read_csv(csv_path, usecols=["omop_measurement_name", "collapsed_measurement"])
    except FileNotFoundError:
        print(f"[warn] OMOP map not found at {csv_path}; RHINO lab names will rely on canonicalize_lab_name only")
        return {}
    mapping = {}
    for _, row in df.iterrows():
        omop = str(row["omop_measurement_name"])
        canonical = str(row["collapsed_measurement"])
        key = _re.sub(r'\s*\[', '__', omop)
        key = _re.sub(r'\]\s*', '__', key)
        key = key.replace('/', '_').replace(' ', '_').rstrip('_')
        mapping[key] = canonical
    return mapping


OMOP_NAME_MAP = _build_omop_name_map(OMOP_MAP_CSV)
print(f"OMOP name map: {len(OMOP_NAME_MAP)} entries loaded")


def _canon_rhino(name: str) -> str:
    """Canonical lab name for a RHINO lab_name string.
    Uses OMOP map first; falls back to helpers.plotting.canonicalize_lab_name."""
    return OMOP_NAME_MAP.get(name, canonicalize_lab_name(name))


_UNIFIED_COLS = [
    "source", "landmark_days", "endpoint", "lab_name", "lab_name_raw",
    "feature_stat", "n_patients", "n_events",
    "coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper",
    "p_value", "q_value", "note",
]


def load_rhino_site(label: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={"landmark_day": "landmark_days"})
    df["lab_name_raw"] = df["lab_name"].copy()
    df["lab_name"] = df["lab_name"].apply(_canon_rhino)
    df["coef_feature"] = np.log(df["hr_per_sd"].clip(lower=1e-9))
    df["hazard_ratio_per_sd"] = df["hr_per_sd"]
    if "n_patients" not in df.columns and "n_events" not in df.columns:
        # column names are already n_patients / n_events in the RHINO schema
        pass
    df["source"] = label
    if "note" not in df.columns:
        df["note"] = ""
    return df[_UNIFIED_COLS]


def load_profile_run(label: str, base_path: Path, landmarks) -> pd.DataFrame:
    parts = []
    for lm in landmarks:
        p = base_path / "cox" / f"landmark_{lm}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
        df = pd.read_csv(p)
        df["landmark_days"] = lm
        parts.append(df)
    df = pd.concat(parts, ignore_index=True)
    df = df.dropna(subset=["coef_feature", "p_value", "q_value"])
    df["lab_name_raw"] = df["lab_name"].copy()
    if "hazard_ratio_per_sd" not in df.columns:
        df["hazard_ratio_per_sd"] = np.exp(df["coef_feature"])
    if "n_patients" not in df.columns:
        df["n_patients"] = df.get("n_patients_used", np.nan)
    if "n_events" not in df.columns:
        df["n_events"] = df.get("n_events_used", np.nan)
    df["source"] = label
    if "note" not in df.columns:
        df["note"] = ""
    return df[_UNIFIED_COLS]


def _try_load(loader, *args, **kwargs):
    try:
        return loader(*args, **kwargs)
    except FileNotFoundError as e:
        print(f"[skip] {args[0]}: file not found — {e.filename}")
        return None


RUNS_RAW = {
    "DFCI":                          _try_load(load_rhino_site,  "DFCI",                          RHINO_DIR / "DFCI_univariate_cox_within.csv"),
    "JHU":                           _try_load(load_rhino_site,  "JHU",                           RHINO_DIR / "JHU_univariate_cox_within.csv"),
    "MSK":                           _try_load(load_rhino_site,  "MSK",                           RHINO_DIR / "MSK_univariate_cox_within.csv"),
    "PROFILE, first ARPI treatment": _try_load(load_profile_run, "PROFILE, first ARPI treatment", PROFILE_ROOT / "local_runs_tx_anchor", LANDMARKS),
}

# Gleason-adjusted runs loaded separately — only the Gleason feature rows are shown
_GLEASON_RUNS_RAW = {
    "Gleason (first-treatment)": _try_load(load_profile_run, "Gleason (first-treatment)", PROFILE_ROOT / "local_runs_gleason",           LANDMARKS),
    "Gleason (first-ARPI)":      _try_load(load_profile_run, "Gleason (first-ARPI)",      PROFILE_ROOT / "local_runs_tx_anchor_gleason", LANDMARKS),
}
_GLEASON_RUNS = {k: v for k, v in _GLEASON_RUNS_RAW.items() if v is not None}

# Drop sources that failed to load
RUNS = {k: v for k, v in RUNS_RAW.items() if v is not None}

print(f"Loaded {len(RUNS)} / {len(RUNS_RAW)} sources: {list(RUNS.keys())}")
for name, df in RUNS.items():
    n_ep = df["endpoint"].unique().tolist()
    n_lm = sorted(df["landmark_days"].unique().tolist())
    print(f"  {name:20s}  {len(df):5d} rows  endpoints={n_ep}  landmarks={n_lm}")

## 5. Stability filter

In [ ]:
def apply_stability_filter(df, coef_cap=2.0, ci_ratio_cap=100):
    ci_ratio = df["ci_upper"] / df["ci_lower"].clip(lower=1e-9)
    mask = (df["coef_feature"].abs() <= coef_cap) & (ci_ratio < ci_ratio_cap)
    n_drop = int((~mask).sum())
    if n_drop:
        print(f"  stability filter: dropping {n_drop} / {len(df)} unstable rows")
    return df.loc[mask].copy()

## 6. Volcano plot helpers

Direct copy of the helpers from `CAIA/generate_figures.ipynb`.

In [ ]:
TOP_K_PER_PANEL         = 6
ALWAYS_LABEL            = {"Hemoglobin", "Albumin", "Alkaline phosphatase"}
LABEL_FORMAT            = "{lab} ({stat})"
LABEL_FONTSIZE          = 8.5
MIN_LABEL_SPACING_NLP   = 0.55
PANEL_XLIM              = (-1.5, 1.5)
Y_MAX_CAP               = 30


def q_threshold_neglog10p(sub):
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    return float(-np.log10(max(float(sig.max()), 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    androgen_rows = sig.loc[sig["category"] == "Androgen axis"]
    non_andro = sig.loc[sig["category"] != "Androgen axis"]
    best = (non_andro.sort_values("p_value")
                     .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = best.loc[~best["lab_name"].isin(always_label)].head(top_k)
    non_andro_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")

    to_label = pd.concat([androgen_rows, non_andro_label]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim  = ax.get_xlim()
    ylim  = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        label_x = xlim[0] + 0.04 * xspan if side == "left" else xlim[1] - 0.04 * xspan
        ha = "left" if side == "left" else "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = min(r["neglog10p"], Y_MAX_CAP)
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text  = label_format.format(lab=r["lab_name"], stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text,
                xy=(r["coef_feature"], min(r["neglog10p"], Y_MAX_CAP)),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    _stack(to_label.loc[to_label["coef_feature"] < 0], "left")
    _stack(to_label.loc[to_label["coef_feature"] >= 0], "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"]   = sub["lab_name"].apply(assign_category)
    sub["neglog10p"]  = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"]        = sub["q_value"] < 0.05
    sub["capped"]     = sub["neglog10p"] > Y_MAX_CAP
    sub["y"]          = sub["neglog10p"].clip(upper=Y_MAX_CAP)

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["y"],
               s=20, color=NS_COLOR, alpha=0.45, edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig  = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero  = cat == "Androgen axis"
        base_kw  = dict(color=CATEGORY_COLORS[cat], alpha=0.92,
                        edgecolor="white", linewidth=0.6)
        in_range = cat_sig.loc[~cat_sig["capped"]]
        if not in_range.empty:
            ax.scatter(in_range["coef_feature"], in_range["y"],
                       s=80 if is_hero else 40, marker="o",
                       zorder=5 if is_hero else 3, **base_kw)
        over = cat_sig.loc[cat_sig["capped"]]
        if not over.empty:
            ax.scatter(over["coef_feature"], over["y"],
                       s=110 if is_hero else 60, marker="^",
                       zorder=6 if is_hero else 4, **base_kw)
            for _, r in over.iterrows():
                ax.annotate(
                    f"p={r['p_value']:.1e}",
                    xy=(r["coef_feature"], Y_MAX_CAP),
                    xytext=(0, -8), textcoords="offset points",
                    ha="center", va="top",
                    fontsize=7.5, color=CATEGORY_COLORS[cat], zorder=7,
                )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["y"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0,    color="grey", linestyle="-",  linewidth=0.7, zorder=0)
    ax.axvline(-0.5, color="grey", linestyle="--", linewidth=0.6, alpha=0.7, zorder=0)
    ax.axvline( 0.5, color="grey", linestyle="--", linewidth=0.6, alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub, top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL, label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig    = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short   = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
               "CMP": "CMP", "Vitals": "Vitals"}
    bd_str  = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")

## 7. Per-source volcano figures

In [ ]:
_legend_handles = [
    Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
    for c in LEGEND_ORDER
]
_legend_handles.append(
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=NS_COLOR, markersize=8, label="ns (q \u2265 0.05)")
)


for label, df in RUNS.items():
    sub = df.loc[df["endpoint"] == ENDPOINT].copy()
    sub = apply_stability_filter(sub)

    panels = [(lm, f"{'+' if lm > 0 else ''}{lm} days") for lm in LANDMARKS]
    fig, axes = plt.subplots(
        1, len(panels), figsize=(7 * len(panels), 6),
        sharex=True, sharey=False,
        gridspec_kw={"wspace": 0.08},
    )
    if len(panels) == 1:
        axes = [axes]

    for ax, (lm, title) in zip(axes, panels):
        panel_sub = sub.loc[sub["landmark_days"] == lm]
        if panel_sub.empty:
            ax.text(0.5, 0.5, f"(no data — landmark {lm}d)",
                    ha="center", va="center", transform=ax.transAxes, color="#7f8c8d")
            ax.set_axis_off()
            continue
        plot_volcano_panel(ax, panel_sub, title)

    fig.legend(handles=_legend_handles, loc="lower center",
               ncol=len(_legend_handles), bbox_to_anchor=(0.5, -0.04), fontsize=10)
    fig.suptitle(f"Univariate Cox — {label}  [{ENDPOINT}]",
                 fontsize=13, weight="bold", y=1.01)

    fname = f"volcano_{label.replace(' ', '_').replace('+', '_')}.png"
    fig.savefig(OUT_DIR / fname)
    print(f"wrote {OUT_DIR / fname}")
    plt.show()

## 8. Androgen-axis deep dive

### 8a. Per-source forest plots (PSA and Testosterone)

For each source: one figure with two panels (PSA left, Testosterone right).
Y-axis = `feature_stat @ {landmark}d`, X-axis = HR on log scale.
Filled circle = q < 0.05; open circle = not significant.

In [ ]:
ANDROGEN_LABS = ["PSA", "Testosterone"]
ANDROGEN_COLOR = CATEGORY_COLORS["Androgen axis"]  # #8e1c2b


def _androgen_forest_panel(ax, df_lab, title, color):
    """Forest panel for one lab within one source."""
    df_lab = df_lab.copy()
    df_lab["row_label"] = df_lab["feature_stat"] + " @ " + df_lab["landmark_days"].astype(str) + "d"
    df_lab = df_lab.sort_values(["landmark_days", "feature_stat"]).reset_index(drop=True)

    if df_lab.empty:
        ax.text(0.5, 0.5, "(no data)", ha="center", va="center",
                transform=ax.transAxes, color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    y   = np.arange(len(df_lab))
    hr  = df_lab["coef_feature"].to_numpy(dtype=float)
    lo  = np.log(df_lab["ci_lower"].clip(lower=1e-9)).to_numpy(dtype=float)
    hi  = np.log(df_lab["ci_upper"].clip(lower=1e-9)).to_numpy(dtype=float)
    sig = (df_lab["q_value"] < 0.05).to_numpy()

    # CI lines
    for i in range(len(df_lab)):
        if np.isfinite(lo[i]) and np.isfinite(hi[i]):
            ax.plot([lo[i], hi[i]], [y[i], y[i]], color=color, lw=1.2, zorder=1)

    # Point estimates: filled = q < 0.05, open = ns
    for mask, fc, lw in [(sig & np.isfinite(hr), color, 0.7),
                         (~sig & np.isfinite(hr), "white", 1.0)]:
        if mask.any():
            ax.scatter(hr[mask], y[mask], s=55, marker="o",
                       facecolor=fc, edgecolor=color, linewidth=lw, zorder=3)

    ax.axvline(0, color="grey", linestyle=":", linewidth=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(df_lab["row_label"], fontsize=9)
    ax.set_xlabel("Cox log HR per SD")
    ax.set_title(title, fontsize=11, weight="bold")


ANDROGEN_SKIP_STATS = {"slope", "n_observations"}

for label, df in RUNS.items():
    andro = df.loc[
        (df["endpoint"] == ENDPOINT) &
        df["lab_name"].isin(ANDROGEN_LABS) &
        ~df["feature_stat"].isin(ANDROGEN_SKIP_STATS)
    ].copy()

    n_labs = sum(andro["lab_name"].eq(lab).any() for lab in ANDROGEN_LABS)
    if n_labs == 0:
        print(f"[skip] {label}: no PSA or Testosterone rows found")
        continue

    fig, axes = plt.subplots(1, len(ANDROGEN_LABS), figsize=(6 * len(ANDROGEN_LABS), 5),
                              constrained_layout=True)
    if len(ANDROGEN_LABS) == 1:
        axes = [axes]

    for ax, lab in zip(axes, ANDROGEN_LABS):
        _androgen_forest_panel(ax, andro.loc[andro["lab_name"] == lab],
                               title=lab, color=ANDROGEN_COLOR)

    fig.suptitle(f"Androgen axis — {label}  [{ENDPOINT}]",
                 fontsize=12, weight="bold")

    fname = f"androgen_persource_{label.replace(' ', '_').replace('+', '_')}.png"
    fig.savefig(OUT_DIR / fname)
    print(f"wrote {OUT_DIR / fname}")
    plt.show()

### 8b. Cross-center comparison

One figure per lab (PSA, Testosterone).
Panels = `feature_stat × landmark_days`.
Within each panel: Y-axis = source, X-axis = HR (log scale).
Color = source; fill = q < 0.05.

In [ ]:
SOURCE_COLORS = {
    "DFCI":                          "#2e86ab",
    "JHU":                           "#a23b72",
    "MSK":                           "#f18f01",
    "PROFILE, first ARPI treatment": "#c73e1d",
}
SOURCE_ORDER = ["DFCI", "JHU", "MSK", "PROFILE, first ARPI treatment"]
# Only include sources that were actually loaded
_active_sources = [s for s in SOURCE_ORDER if s in RUNS]

# Collect all unique (feature_stat, landmark_days) panels across loaded runs
_all_andro = pd.concat([
    df.loc[
        (df["endpoint"] == ENDPOINT) &
        df["lab_name"].isin(ANDROGEN_LABS) &
        ~df["feature_stat"].isin(ANDROGEN_SKIP_STATS)
    ]
    for df in RUNS.values()
], ignore_index=True)

_stat_lm_combos = (
    _all_andro[["feature_stat", "landmark_days"]]
    .drop_duplicates()
    .sort_values(["landmark_days", "feature_stat"])
    .values.tolist()
)


def _cross_center_figure(lab_name):
    n_panels = len(_stat_lm_combos)
    n_cols   = 5
    n_rows   = 2
    n_src    = len(_active_sources)
    panel_h  = max(1.0, 0.45 * n_src)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.5 * n_cols, panel_h * n_rows + 1.0),
        constrained_layout=True,
    )
    axes_flat = np.array(axes).flatten() if n_panels > 1 else [axes]

    for ax_idx, (stat, lm) in enumerate(_stat_lm_combos):
        ax = axes_flat[ax_idx]
        y_ticks   = []
        y_labels  = []

        for y_pos, src in enumerate(_active_sources):
            src_df = RUNS[src]
            row = src_df.loc[
                (src_df["endpoint"] == ENDPOINT) &
                (src_df["lab_name"] == lab_name) &
                (src_df["feature_stat"] == stat) &
                (src_df["landmark_days"] == lm)
            ]
            y_ticks.append(y_pos)
            y_labels.append(src)

            if row.empty:
                ax.scatter([0.0], [y_pos], s=12, marker="x",
                           color="#d5d8dc", zorder=1)
                continue

            hr   = float(row.iloc[0]["coef_feature"])
            lo   = float(np.log(max(row.iloc[0]["ci_lower"], 1e-9)))
            hi   = float(np.log(max(row.iloc[0]["ci_upper"], 1e-9)))
            q    = float(row.iloc[0]["q_value"])
            col  = SOURCE_COLORS.get(src, "#7f8c8d")

            if np.isfinite(lo) and np.isfinite(hi):
                ax.plot([lo, hi], [y_pos, y_pos], color=col, lw=1.2, zorder=1)
            if np.isfinite(hr):
                fc = col if q < 0.05 else "white"
                ax.scatter([hr], [y_pos], s=50, marker="o",
                           facecolor=fc, edgecolor=col, linewidth=1.0, zorder=3)

        ax.axvline(0, color="grey", linestyle=":", linewidth=0.9)
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_labels, fontsize=8.5)
        sign = "+" if lm > 0 else ""
        ax.set_title(f"{stat}  ·  {sign}{lm}d", fontsize=9.5, weight="bold")
        ax.set_xlabel("Cox log HR per SD", fontsize=8)

    # Hide unused panels
    for ax in axes_flat[n_panels:]:
        ax.set_visible(False)

    # Source color legend
    legend_handles = [
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor=SOURCE_COLORS.get(s, "#7f8c8d"),
               markersize=8, label=s)
        for s in _active_sources
    ]
    legend_handles += [
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor="#7f8c8d", markersize=8, label="filled = q<0.05"),
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor="white", markeredgecolor="#7f8c8d",
               markersize=8, label="open = ns"),
    ]
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=min(len(legend_handles), 5),
               bbox_to_anchor=(0.5, -0.04), fontsize=9)

    fig.suptitle(f"{lab_name} — cross-center comparison  [{ENDPOINT}]",
                 fontsize=13, weight="bold")
    return fig


for lab in ANDROGEN_LABS:
    fig = _cross_center_figure(lab)
    fname = f"androgen_cross_center_{lab.replace(' ', '_')}.png"
    fig.savefig(OUT_DIR / fname)
    print(f"wrote {OUT_DIR / fname}")
    plt.show()

In [ ]:
## 9. Gleason group-specific results
#
# From the Gleason-adjusted PROFILE runs, extract only the rows where
# the predictor is the Gleason/ISUP grade group feature itself.

import re as _re2
_GLEASON_PAT = _re2.compile(r'gleason|isup', _re2.IGNORECASE)
_DISPLAY_COLS = [
    "source", "landmark_days", "feature_stat", "lab_name_raw",
    "n_patients", "n_events",
    "coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper",
    "p_value", "q_value",
]

for _label, _df in _GLEASON_RUNS.items():
    _rows = _df.loc[
        (_df['endpoint'] == ENDPOINT) &
        _df['lab_name_raw'].str.contains(_GLEASON_PAT, na=False)
    ].copy()
    print(f'\n========== {_label} ==========')
    if _rows.empty:
        # Fall back to lab_name if raw didn't match
        _rows = _df.loc[
            (_df['endpoint'] == ENDPOINT) &
            _df['lab_name'].str.contains(_GLEASON_PAT, na=False)
        ].copy()
    if _rows.empty:
        print('  (no Gleason/ISUP rows found — check lab_name_raw values below)')
        print(_df[['lab_name_raw', 'lab_name']].drop_duplicates().head(10).to_string(index=False))
        continue
    _show = [c for c in _DISPLAY_COLS if c in _rows.columns]
    print(_rows[_show].sort_values(['landmark_days', 'feature_stat']).to_string(index=False))
